### Import Libraries

In [ ]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import text, create_engine
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify
from shapely.geometry import LineString

### Import Data

In [ ]:
# Connection with the DB
engine = create_engine("postgresql://mobiledata:@cirrus.ita.chalmers.se/se_mobile_data",
                       connect_args={"client_encoding":
                                                                                                  "utf8"})
conn = engine.connect()

In [ ]:
# Import Data
# howde_from_stops_2024
# City name
city = 'Simrishamn'   # <-- just change this
city_l = 'Simrishamn'

# home
howde_home_2024_query = text(
    f'''
    SELECT DISTINCT ON (device_uid)
        device_uid,
        location_type,
        date,
        geom
    FROM (
        SELECT
            device_uid,
            location_type,
            date,
            geom,
            COUNT(*) OVER (PARTITION BY device_uid, location_type, date, geom) AS geom_count
        FROM extracts.srh_proto_tourism_howde_2024_city
        WHERE location_type = 'H'
    ) sub
    ORDER BY device_uid, geom_count DESC;
    '''
)

howde_home_2024 = (
    gpd.read_postgis(howde_home_2024_query, engine, geom_col='geom')
      .to_crs(epsg=4326)
)

# work
howde_work_2024_query = text(
    f'''
    SELECT DISTINCT ON (device_uid)
        device_uid,
        location_type,
        date,
        geom
    FROM (
        SELECT
            device_uid,
            location_type,
            date,
            geom,
            COUNT(*) OVER (PARTITION BY device_uid, location_type, date,geom) AS geom_count
        FROM extracts.srh_proto_tourism_howde_2024_city
        WHERE location_type = 'W'
    ) sub
    ORDER BY device_uid, geom_count DESC;
    '''
)

howde_work_2024 = (
    gpd.read_postgis(howde_work_2024_query, engine, geom_col='geom')
      .to_crs(epsg=4326)
)

# work
howde_other_2024_query = text(
    f'''
    SELECT DISTINCT ON (device_uid)
        device_uid,
        location_type,
        date,
        geom
    FROM (
        SELECT
            device_uid,
            location_type,
            date,
            geom,
            COUNT(*) OVER (PARTITION BY device_uid, location_type, date, geom) AS geom_count
        FROM extracts.srh_proto_tourism_howde_2024_city
        WHERE location_type = 'O'
    ) sub
    ORDER BY device_uid, geom_count DESC;
    '''
)

howde_other_2024 = (
    gpd.read_postgis(howde_other_2024_query, engine, geom_col='geom')
      .to_crs(epsg=4326)
)

# boundaries
deso_2025_query = text('SELECT * FROM boundaries.deso_2025_v1')
deso_2025 = (
    gpd.read_postgis(deso_2025_query, engine, geom_col='geom').set_crs(epsg=3006)
    .to_crs(epsg=4326)
)

In [ ]:
# stops
stops_query = text('SELECT * FROM extracts.srh_proto_tourism_infostops_stops_2024_city_10min')
stops = (
    gpd.read_postgis(stops_query, engine, geom_col='geometry')
).rename(columns={'geometry': 'geom'}).set_geometry('geom')

In [ ]:
# boundaries
srh_boundary_query = text('SELECT * FROM boundaries.srh_city_boundary_4326')
srh_boundary = (
    gpd.read_postgis(srh_boundary_query, engine, geom_col='geom')
)

### Preprocess the data

#### Make the data compatible

In [ ]:
# Normalize both columns to strings and make consistent
def normalize_device_uid(df, column='device_uid'):
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    return df

# apply
howde_home_2024 = normalize_device_uid(howde_home_2024)
howde_work_2024 = normalize_device_uid(howde_work_2024)
howde_other_2024 = normalize_device_uid(howde_other_2024)

#### Spatial Join

In [ ]:
# Spatial join
def spatial_join_howde_with_deso(howde_df, deso_df):
    # Keep only relevant columns
    howde_sub = howde_df[['device_uid', 'location_type', 'date', 'geom']].copy()
    deso_df = (deso_df[['desokod', 'kommunnamn', 'geom']])

    # Set active geometry
    howde_sub = howde_sub.set_geometry('geom')
    deso_df = deso_df.set_geometry('geom')

    # --- Ensure CRS match ---
    if howde_sub.crs != deso_df.crs:
        deso_df = deso_df.to_crs(howde_sub.crs)

    # Perform spatial join
    joined = gpd.sjoin(
        howde_sub,
        deso_df,
        predicate='intersects',
        how='right'
    ).drop(columns=['index_left'], errors='ignore').drop_duplicates(['device_uid', 'desokod'])

    # Rename columns
    #joined = joined.rename(columns={'deso': 'deso_code', 'totalt': 'work_pop_grid', 'municipal_name': 'city_name'})

    # Drop rows without device_uid
    joined = joined.dropna(subset=['device_uid'])

    return joined

# apply
howde_home_deso_2024 = spatial_join_howde_with_deso(howde_home_2024, deso_2025)
howde_other_deso_2024 = spatial_join_howde_with_deso(howde_other_2024, deso_2025)
howde_work_deso_2024 = spatial_join_howde_with_deso(howde_work_2024, deso_2025)

### Create Connections, Graphs

#### Merge work and homes to create the graphs

In [ ]:
def merge_other_home_work(homes_df, work_df, other_df):
    cols = ['device_uid', 'location_type', 'date', 'desokod', 'kommunnamn', 'geom']

    # Other → Home
    other_home = other_df[cols].merge(
        homes_df[cols],
        on='device_uid',
        how='inner',
        suffixes=('_other', '_home')
    )

    # Other/Home → Work
    other_home_work = other_home.merge(
        work_df[cols],
        on='device_uid',
        how='inner'
    )

    # Rename work columns for consistency
    other_home_work = other_home_work.rename(columns={
        'location_type': 'location_type_work',
        'date': 'date_work',
        'desokod': 'desokod_work',
        'kommunnamn': 'kommunnamn_work',
        'geom': 'geom_work'
    })

    return other_home_work


combined_howde = merge_other_home_work(
    howde_home_deso_2024,
    howde_work_deso_2024,
    howde_other_deso_2024
).drop(columns=['date_home', 'geom_other'])

combined_howde

In [ ]:
combined_howde = combined_howde[
    (combined_howde['kommunnamn_other'] == 'Simrishamn') &
    (combined_howde['kommunnamn_home'] != 'Simrishamn') &
    (combined_howde['kommunnamn_work'] != 'Simrishamn')
]

combined_howde

In [ ]:
home_table = combined_howde[
    [
        'device_uid',
        'desokod_other',
        'kommunnamn_other',
        'desokod_home',
        'kommunnamn_home',
        'geom_home'
    ]
].copy()

# Other -> Work table
work_table = combined_howde[
    [
        'device_uid',
        'desokod_other',
        'kommunnamn_other',
        'desokod_work',
        'kommunnamn_work',
        'geom_work'
    ]
].copy()

home_table.head()
work_table.head()

In [ ]:
# Home counts by DeSO with geometry and municipality name
home_counts = (
    home_table
    .groupby('desokod_home')
    .agg(
        unique_devices=('device_uid', 'nunique'),
        kommunnamn_other=('kommunnamn_home', 'first'),
        geometry=('geom_home', 'first')
    )
    .reset_index()
)

home_counts = gpd.GeoDataFrame(
    home_counts,
    geometry='geometry'
)

work_counts = (
    work_table
    .groupby('desokod_work')
    .agg(
        unique_devices=('device_uid', 'nunique'),
        kommunnamn_other=('kommunnamn_work', 'first'),
        geometry=('geom_work', 'first')
    )
    .reset_index()
)

work_counts = gpd.GeoDataFrame(
    work_counts,
    geometry='geometry',
)

In [ ]:
home_counts

In [ ]:
work_counts

In [ ]:
howde_other_2024_filtered = stops[
    stops['device_uid'].isin(
        combined_howde['device_uid']
    )
].copy()
howde_other_2024_filtered

In [ ]:
stops_joined = gpd.sjoin(
        howde_other_2024_filtered,
        srh_boundary,
        predicate='intersects',
        how='inner'
    )

In [ ]:
stops_joined

In [ ]:
output_path = (
    "C:/Users/monaliza/Chalmers/Chalmers.CNL-Urban Analytics Lab - "
    "General/Prototypes/Proto_260529/overtourism_test.gpkg"
)

# Make sure all layers are in EPSG:4326 before exporting
stops_joined_4326 = stops_joined.to_crs(epsg=4326)
home_counts = home_counts.set_crs(epsg=3006, allow_override=True)
work_counts = work_counts.set_crs(epsg=3006, allow_override=True)

home_counts_4326 = home_counts.to_crs(epsg=4326)
work_counts_4326 = work_counts.to_crs(epsg=4326)

stops_joined_4326.to_file(
    output_path,
    layer="tourists_stops_10min",
    driver="GPKG"
)

home_counts_4326.to_file(
    output_path,
    layer="tourists_home_deso",
    driver="GPKG"
)

work_counts_4326.to_file(
    output_path,
    layer="tourists_work_deso",
    driver="GPKG"
)